# Ferramentas

Um modelo de linguagem produz tokens, e nada além disso. Calcular com exatidão, ler um arquivo ou consultar um serviço acontece fora dele, em código que alguém escreveu.

Ferramenta é o arranjo que liga as duas coisas, em quatro passos: **declarar** as funções disponíveis no prompt, **pedir** a execução em um formato conhecido, **executar** a função em Python e **devolver** o resultado no contexto. O modelo escreve texto nos passos ímpares; o programa age nos pares.

Nada nesse arranjo é capacidade do modelo. O formato de chamada é convenção textual, definida no prompt ou aprendida no ajuste, e pedir a chamada é uma escolha de geração, sujeita a erro como qualquer outra: a ferramenta certa pode não ser chamada, a errada pode ser, e os argumentos podem vir fora do tipo.

In [ ]:
# No Google Colab, descomente e rode uma vez (Ambiente de execução > GPU).
# !pip install -q "agentkit @ git+https://github.com/silvaan/agentic-ai"

import inspect
import json
import urllib.parse
import urllib.request
from pathlib import Path
from typing import Callable, get_type_hints

import pandas as pd
import torch

from agentkit import LLM

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
llm = LLM(MODEL_NAME, device=device, temperature=0.0, max_tokens=200)
print(llm.model)

## Falhas sem ferramenta

As duas células seguintes pedem coisas que parecem simples. Antes de rodar, tente prever o resultado de cada uma.

In [ ]:
print(llm.invoke([{"role": "user", "content": "Quanto é 4871 vezes 3926? Responda apenas com o número."}]))
print(f"resposta correta: {4871 * 3926}")

A multiplicação exata exige um algoritmo com transporte entre casas. O modelo prevê tokens plausíveis para um número dessa forma, e acerta a quantidade de dígitos com o valor errado.

In [ ]:
print(llm.invoke([
    {"role": "user", "content": "Salve um resumo de uma linha sobre a linguagem Python em um arquivo chamado resumo.txt."},
], max_tokens=100))
print(f"arquivo existe: {Path('resumo.txt').exists()}")

O modelo devolve o código que resolveria o pedido, e nenhum arquivo aparece, porque nada foi executado. Os dois casos têm a mesma causa: a saída é texto, e texto não multiplica nem grava em disco.

## Protocolo de chamada em texto

O formato de chamada não é uma capacidade do modelo: é uma convenção de texto definida no prompt. Esta parte define uma convenção simples, para que o mecanismo fique visível antes de ser encapsulado.

A convenção precisa de três coisas: quais funções existem, como pedir uma delas e o que fazer quando o resultado chegar.

In [ ]:
SYSTEM = """Você é um assistente com acesso a ferramentas.

# Ferramentas
calculate(expression: str): avalia uma expressão aritmética e devolve o resultado.

# Regras
Você não faz contas de cabeça: toda conta é feita pela ferramenta.
Para chamar a ferramenta, responda apenas com uma linha, sem nenhum outro texto:
CHAMAR: {"name": "calculate", "arguments": {"expression": "..."}}"""

question = "Quanto é 4871 vezes 3926?"
messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]

answer = llm.invoke(messages, max_tokens=80)
print(answer)

O modelo respondeu com a linha combinada, no lugar do número inventado da primeira parte. Esse é o passo **pedir**: existe uma intenção de chamada escrita em texto, e nada foi executado.

O passo **executar** é código comum: ler a linha, virar dicionário, chamar a função.

In [ ]:
def parse_call(text: str) -> dict | None:
    """Lê a linha CHAMAR e devolve o dicionário da chamada, ou None se não houver."""
    for line in text.splitlines():
        if line.strip().startswith("CHAMAR:"):
            try:
                return json.loads(line.split("CHAMAR:", 1)[1])
            except json.JSONDecodeError:
                return None
    return None


call = parse_call(answer)
# O modelo escreve essa string, então a avaliação roda sem acesso a builtins.
result = str(eval(call["arguments"]["expression"], {"__builtins__": {}}, {}))
print(call, "->", result)

Falta **devolver**. O resultado entra na conversa como uma mensagem nova, com o papel `tool`, e o turno do assistente que pediu a chamada fica registrado antes dela.

In [ ]:
messages += [
    {"role": "assistant", "content": answer},
    {"role": "tool", "name": call["name"], "content": result},
]

rendered = llm.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(rendered[rendered.index("<|im_start|>assistant") :])

O papel `tool` existe na lista de mensagens e não no prompt: o template o converte em um turno de usuário com o conteúdo dentro de `<tool_response>`. A lista de dicionários é a estrutura de programação; o texto renderizado é o que o modelo lê.

In [ ]:
print(llm.invoke(messages, max_tokens=80))

O ciclo fechou e a resposta usa o número que veio da execução. Tudo que existe aqui é prompt, string e uma chamada de função.

### Descrição gerada da assinatura

Escrever a descrição direto no prompt duplica o que a função já declara: nome, parâmetros e tipos estão na assinatura, e o resto está na docstring. O decorador lê isso por introspecção e anexa o esquema à função.

In [ ]:
TYPE_NAMES = {str: "string", int: "integer", float: "number", bool: "boolean"}


def tool(fn: Callable) -> Callable:
    """Anexa fn.tool_schema por introspecção e devolve a própria função."""
    description = inspect.getdoc(fn)
    if not description:
        raise ValueError(f"A ferramenta {fn.__name__} precisa de docstring.")
    hints = get_type_hints(fn)
    names = list(inspect.signature(fn).parameters)
    fn.tool_schema = {
        "name": fn.__name__,
        "description": description,
        "parameters": {name: {"type": TYPE_NAMES[hints[name]]} for name in names},
        "required": names,
    }
    return fn

Duas decisões merecem atenção. O decorador recusa função sem docstring, porque a descrição é o que o modelo lê para escolher. E a tabela de tipos tem quatro entradas de propósito: um parâmetro de tipo não previsto levanta `KeyError` na linha do `@tool`, no import, e não em produção com o modelo esperando resposta.

A docstring tem dois leitores, o programador e o modelo. É o segundo que decide o idioma: ela é copiada para dentro do prompt, então acompanha a língua da conversa.

In [ ]:
@tool
def calculate(expression: str) -> str:
    """Avalia uma expressão aritmética, como 12 * (3 + 4)."""
    return str(eval(expression, {"__builtins__": {}}, {}))


print(json.dumps(calculate.tool_schema, indent=2, ensure_ascii=False))
print(calculate("4871 * 3926"))

O decorador devolve a própria função, que continua chamável e testável sem modelo nenhum. A avaliação sem builtins bloqueia `open`, `__import__` e o resto da biblioteca padrão a partir da string: é a proteção mínima para executar algo que veio do modelo, e uma ferramenta de produção usaria um analisador de expressões no lugar de `eval`.

## Ferramentas ligadas ao modelo

A convenção da parte anterior foi definida por nós no prompt de sistema. Modelos ajustados para ferramentas trazem uma convenção própria, aprendida no treino, e o template de conversa a escreve a partir dos esquemas passados em `tools`.

In [ ]:
prompt = llm.tokenizer.apply_chat_template(
    [{"role": "user", "content": question}],
    tools=[calculate.tool_schema], tokenize=False, add_generation_prompt=True,
)
print(prompt)

O template escreveu o esquema em um bloco `<tools>` na mensagem de sistema e mandou responder com um objeto JSON entre `<tool_call>` e `</tool_call>`. O retorno continua no papel `tool`, como na parte anterior. Mudam os marcadores e quem os escreve; a estrutura é a mesma.

In [ ]:
print(llm.generate(prompt, max_tokens=120))

Renderizar as mensagens com os esquemas e ler os blocos `<tool_call>` da resposta são dois passos que se repetem a cada chamada e não dependem da tarefa. O `agentkit` os encapsula em `LLM.bind_tools`, que devolve uma cópia do modelo com as ferramentas ligadas. Daqui em diante basta chamar `invoke` nessa cópia.

In [ ]:
llm_with_tools = llm.bind_tools([calculate])

llm_with_tools.invoke([{"role": "user", "content": question}])

Com ferramentas ligadas, `invoke` devolve a mensagem do assistente em vez do texto, e a chamada vem normalizada em uma lista de dicionários com `name` e `arguments`. Quando o modelo responde sem pedir ferramenta, a mensagem vem só com `content`, e é a ausência de `tool_calls` que encerra o laço. O modelo original não muda: `llm.invoke` continua devolvendo texto.

O método vive em `model.py`, e não em `tools.py`, porque o protocolo é do modelo: quem sabe escrever `<tool_call>` é o template deste modelo. Repare também que as ferramentas aparecem duas vezes no programa, ligadas ao modelo para serem declaradas e passadas ao laço para serem executadas. Declarar e executar são coisas diferentes, e `bind_tools` cuida só da primeira.

### Execução da chamada

Executar é o que falta. Com a chamada normalizada, o despacho é uma tabela de nome para função.

In [ ]:
def run_tool(call: dict, tools: dict) -> str:
    """Executa a ferramenta pedida e devolve o resultado como texto."""
    fn = tools.get(call["name"])
    if fn is None:
        return f"ferramenta desconhecida: {call['name']}"
    try:
        return str(fn(**call["arguments"]))
    except Exception as error:
        # A exceção vira observação, porque o modelo precisa poder reagir a ela.
        return f"{type(error).__name__}: {error}"

A entrada é o dicionário que veio em `tool_calls`, mais a tabela das ferramentas disponíveis; a saída é sempre texto, porque ela vai entrar na conversa como observação. A célula abaixo cobre os três desfechos possíveis.

In [ ]:
available = {"calculate": calculate}

print(run_tool({"name": "calculate", "arguments": {"expression": "37 * 42"}}, available))
print(run_tool({"name": "calculate", "arguments": {"expression": "1 / 0"}}, available))
print(run_tool({"name": "soma", "arguments": {"a": 1, "b": 2}}, available))

O primeiro é o caso normal. O segundo é uma expressão que levanta exceção, e o terceiro é um nome que não existe na tabela: os dois viram texto descrevendo a falha, e nenhum deles interrompe a execução.

É a única captura ampla de exceção do notebook, e existe porque erro de ferramenta é caso normal do laço. O nome também vem do modelo, então errar o nome é tão esperado quanto errar o argumento. Quem decide o que fazer com a falha é o modelo, no passo seguinte, lendo a observação.

### O agente

As ferramentas entram no agente de duas formas, e as duas são decididas na montagem: ligadas ao modelo, para serem declaradas, e em uma tabela de despacho, para serem executadas. O agente guarda as duas peças e o limite de passos.

In [ ]:
class Agent:
    """Modelo com ferramentas ligadas e o laço que alterna chamada e execução."""

    def __init__(self, llm: LLM, tools: list[Callable], max_steps: int = 5) -> None:
        self.llm = llm.bind_tools(tools)
        self.tools = {fn.tool_schema["name"]: fn for fn in tools}
        self.max_steps = max_steps

    def run(self, input: str | list[dict]) -> list[dict]:
        """Responde à pergunta, ou continua a conversa, e devolve o histórico."""
        messages = [{"role": "user", "content": input}] if isinstance(input, str) else list(input)
        for _ in range(self.max_steps):
            message = self.llm.invoke(messages)
            messages.append(message)
            if "tool_calls" not in message:
                return messages
            for call in message["tool_calls"]:
                observation = run_tool(call, self.tools)
                messages.append({"role": "tool", "name": call["name"], "content": observation})
        return messages + [{"role": "assistant", "content": "limite de passos atingido"}]

O laço não menciona template, marcador nem parse: pede a mensagem, executa o que ela pedir, devolve a observação e repete. O limite de passos é a proteção contra laço infinito.

O `run` aceita as duas formas que o `invoke` do modelo aceita. Uma string vira a pergunta do usuário; uma lista de mensagens entra como está, que é o caminho para acrescentar uma mensagem `system` com a política da aplicação ou para continuar a conversa a partir do histórico devolvido antes. O agente não guarda esse histórico: ele entra e sai a cada chamada, e é o programa que decide o que fazer com ele.

In [ ]:
for message in Agent(llm, [calculate]).run("Quanto é 4871 vezes 3926?"):
    print(message["role"], ":", message.get("content") or message["tool_calls"])

Quatro mensagens: a pergunta, o turno que pede a chamada, a observação e a resposta final.

### Exercício 1

`DATABASE` é um banco de dados falso: um dicionário no lugar de um SGBD, com o cadastro de três clientes. Escreva duas ferramentas, uma que consulta o cadastro pelo e-mail e outra que acrescenta créditos, decidindo nome, parâmetros, tipos, docstring e o que cada uma devolve quando o e-mail não está no banco.

Monte o agente e rode os quatro pedidos. Responda se o modelo escolheu a ferramenta certa em cada um, o que ele fez com o e-mail inexistente, e como ficou o `DATABASE` no fim, que é a diferença prática entre uma ferramenta que lê e uma que escreve.

In [ ]:
# Banco de dados falso: um dicionário no lugar de um SGBD.
DATABASE = {
    "ana@acme.com": {"nome": "Ana Lima", "plano": "atlas", "creditos": 120},
    "bruno@acme.com": {"nome": "Bruno Sá", "plano": "orbit", "creditos": 0},
    "dora@acme.com": {"nome": "Dora Reis", "plano": "atlas", "creditos": 45},
}

DATABASE_REQUESTS = [
    "Quantos créditos a Ana tem? O e-mail dela é ana@acme.com.",
    "Adicione 50 créditos para bruno@acme.com.",
    "Consulte o cadastro de carla@acme.com.",
    "Dê 10 créditos para dora@acme.com e me diga o saldo final dela.",
]

# Seu código aqui

### Escrita e leitura de arquivos

Ferramenta com efeito colateral precisa de fronteira. As duas abaixo trabalham dentro de um diretório de trabalho, e nomes que tentem sair dele são recusados antes de qualquer acesso.

In [ ]:
WORKSPACE = Path("workspace")


def resolve(name: str) -> Path:
    """Resolve o nome dentro do diretório de trabalho e recusa caminhos fora dele."""
    WORKSPACE.mkdir(exist_ok=True)
    target = (WORKSPACE / name).resolve()
    if not target.is_relative_to(WORKSPACE.resolve()):
        raise ValueError("caminho fora do diretório de trabalho")
    return target

In [ ]:
@tool
def write_file(name: str, content: str) -> str:
    """Escreve um texto em um arquivo do diretório de trabalho."""
    path = resolve(name)
    path.write_text(content, encoding="utf-8")
    return f"{len(content)} caracteres escritos em {name}"


@tool
def read_file(name: str) -> str:
    """Lê um arquivo do diretório de trabalho e devolve o conteúdo. Use esta ferramenta sempre que o usuário perguntar o que há em um arquivo."""
    return resolve(name).read_text(encoding="utf-8")

In [ ]:
file_agent = Agent(llm, [write_file, read_file])

history = file_agent.run(
    "Salve um resumo de uma linha sobre a linguagem Python em um arquivo chamado resumo.txt."
)
print(history[-1]["content"])
print(WORKSPACE.joinpath("resumo.txt").read_text(encoding="utf-8"))

O arquivo existe no disco e o conteúdo foi escrito pelo modelo. É a segunda falha da abertura resolvida, e a primeira vez que uma resposta produz efeito fora do processo.

In [ ]:
history = file_agent.run("O que está escrito no arquivo resumo.txt?")
print(history[-1]["content"])

### Exercício 2

Agora o agente escreve código, e código tem uma vantagem sobre texto: dá para executar e conferir. As ferramentas de arquivo já existem, então o trabalho aqui é montar o agente, escrever o pedido de modo que o arquivo salvo seja Python executável, e verificar o resultado.

`check_fibonacci` roda o arquivo gerado e compara os dez primeiros termos. Chame-a depois do agente e responda o que aconteceu: se o script executou, se os valores batem e o que veio no arquivo além da função. Se tiver falhado, diga qual mensagem de erro você devolveria ao agente para que ele tentasse de novo.

In [ ]:
FIBONACCI_REQUEST = (
    "Escreva um script Python com uma função fibonacci(n) que devolve o n-ésimo número "
    "de Fibonacci, com fibonacci(0) igual a 0, e salve em fibonacci.py."
)


def check_fibonacci() -> None:
    """Executa o arquivo gerado e compara com os dez primeiros termos."""
    expected = [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
    namespace: dict = {}
    exec(WORKSPACE.joinpath("fibonacci.py").read_text(encoding="utf-8"), namespace)
    obtained = [namespace["fibonacci"](n) for n in range(10)]
    print("obtido: ", obtained)
    print("esperado:", expected)
    print("passou" if obtained == expected else "falhou")


# Seu código aqui

### Consulta a uma API

A ferramenta abaixo consulta um serviço público de previsão do tempo, sem chave de acesso. Ela traz para o contexto um dado que não está nos pesos e que muda a cada hora.

In [ ]:
@tool
def get_temperature(latitude: float, longitude: float) -> str:
    """Consulta a temperatura atual, em graus Celsius, de uma latitude e uma longitude."""
    url = (
        "https://api.open-meteo.com/v1/forecast"
        f"?latitude={latitude}&longitude={longitude}&current=temperature_2m"
    )
    with urllib.request.urlopen(url, timeout=10) as response:
        data = json.load(response)
    return f"{data['current']['temperature_2m']} C"

In [ ]:
history = Agent(llm, [get_temperature]).run(
    "Qual é a temperatura atual em Natal, no Brasil? As coordenadas são -5.79, -35.21."
)
print(history[-2]["content"])
print(history[-1]["content"])

A penúltima mensagem é a observação crua do serviço, e a última é a resposta escrita a partir dela. Uma ferramenta assim acrescenta três problemas que as anteriores não têm: latência variável, falha fora do controle do programa e resposta que muda entre execuções, o que impede comparar duas rodadas por igualdade de texto.

### Exercício 3

A Wikipédia tem uma API pública, sem cadastro e sem chave. `wikipedia_api` faz a requisição e devolve o artigo encontrado para um termo de busca; falta transformá-la em ferramenta, e as decisões são suas: nome, parâmetro, tipo, docstring e quanto do artigo devolver, lembrando que o texto completo passa de mil caracteres e que a observação inteira entra no contexto a cada chamada.

Monte o agente e rode as quatro perguntas; a última não tem artigo. Depois repita as duas primeiras sem a palavra "pesquise" e responda quantas viraram busca em cada versão, e o que a diferença diz sobre quem decide usar a ferramenta.

In [ ]:
def wikipedia_api(term: str) -> str:
    """Busca o termo na Wikipédia em português e devolve o início do artigo."""
    url = (
        "https://pt.wikipedia.org/w/api.php?action=query&format=json"
        "&prop=extracts&exintro=1&explaintext=1&redirects=1"
        "&generator=search&gsrlimit=1&gsrsearch=" + urllib.parse.quote(term)
    )
    request = urllib.request.Request(url, headers={"User-Agent": "agentkit-aula/0.1"})
    with urllib.request.urlopen(request, timeout=10) as response:
        data = json.load(response)
    pages = data.get("query", {}).get("pages")
    if not pages:
        return f"nada encontrado para {term}"
    page = next(iter(pages.values()))
    return f"{page['title']}: {page['extract']}"


SEARCH_QUESTIONS = [
    "Pesquise o que é o mecanismo de atenção multicabeça.",
    "Procure quem foi Ada Lovelace e me diga em uma frase.",
    "Pesquise o que é a Universidade Federal do Rio Grande do Norte.",
    "Pesquise o que é xyzzy plugh 12345.",
]

# Seu código aqui

## Chamadas paralelas e dependência entre passos

O agente até aqui usou uma ferramenta por vez. Com quatro declaradas, aparecem os dois casos que decidem o comportamento de um laço com ferramentas.

In [ ]:
agent = Agent(llm, [calculate, get_temperature, write_file, read_file])

for message in agent.run("Qual é a temperatura em Natal (-5.79, -35.21) e quanto é 37 * 42?"):
    print(message["role"], ":", message.get("content") or message["tool_calls"])

O modelo emitiu as duas chamadas no mesmo turno, e o laço executou as duas antes de devolver as observações. É o caso fácil: as duas são escritas antes de qualquer resultado existir, o que só funciona porque uma não depende da outra.

A pergunta seguinte quebra essa condição, porque o texto a gravar é o resultado da consulta. Antes de rodar, tente prever o que o modelo faz.

In [ ]:
for message in agent.run("Consulte a temperatura em Natal (-5.79, -35.21) e salve o valor no arquivo clima.txt."):
    print(message["role"], ":", message.get("content") or message["tool_calls"])

Com dependência entre os passos, as saídas ruins são duas: preencher o argumento por conta própria, como se o resultado já fosse conhecido, ou tratar o pedido como se fosse só a primeira parte e não voltar para a segunda. Leia o traço e veja qual das duas aconteceu.

O laço não tem como corrigir isso: ele executa o que for pedido e devolve a observação. Decidir a sequência de passos antes de agir, e revisar o que já foi feito, é assunto da aula de planejamento e reflexão.

As três peças deste notebook existem no `agentkit`, divididas pela mesma lógica. O decorador e a execução ficam em `tools.py`, junto de um protocolo textual como o da segunda parte, que mantém o pacote utilizável com modelos não ajustados para ferramentas. `bind_tools` fica em `model.py`, porque o protocolo do template pertence ao modelo. O laço fica em `agent.py`, com limite de passos, orçamento de tokens e traço, na forma de uma função que recebe o modelo e as ferramentas.